In [1]:
import os
import numpy as np
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from transformers import ViTForImageClassification
from transformers import ViTImageProcessor
from transformers import TrainingArguments
from transformers import Trainer

from datasets import Dataset as HFDataset, Features, ClassLabel, Array3D
import evaluate

In [2]:
DATA_DIR = "/home/jupyter/project/tiny"  # путь к папке с данными

def load_dataset_paths(data_dir):
    """Загружает пути к изображениям и метки"""
    image_paths = []
    labels = []
    class_names = sorted(os.listdir(data_dir))  # ['A', 'B', ..., 'nothing', 'space']

    print(f"Найдено классов: {len(class_names)}")
    print(f"Классы: {class_names}")

    for label_idx, class_name in enumerate(class_names):
        class_dir = os.path.join(data_dir, class_name)
        if os.path.isdir(class_dir):
            for img_file in os.listdir(class_dir):
                if img_file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    image_paths.append(os.path.join(class_dir, img_file))
                    labels.append(label_idx)

    print(f"Всего изображений: {len(image_paths)}")
    return image_paths, labels, class_names

image_paths, labels, class_names = load_dataset_paths(DATA_DIR)

# Разделение на train/val/test
train_paths, test_paths, train_labels, test_labels = train_test_split(
    image_paths, labels, test_size=0.2, random_state=42, stratify=labels
)
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_paths, train_labels, test_size=0.1, random_state=42, stratify=train_labels
)

print(f"Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}")


Найдено классов: 28
Классы: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'nothing', 'space']
Всего изображений: 14000
Train: 10080 | Val: 1120 | Test: 2800


In [3]:
MODEL_NAME = "google/vit-base-patch16-224"

# Загрузка процессора (обработка изображений)
processor = ViTImageProcessor.from_pretrained(MODEL_NAME)

# Загрузка модели с нужным количеством классов
model = ViTForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(class_names),
    id2label={i: name for i, name in enumerate(class_names)},
    label2id={name: i for i, name in enumerate(class_names)},
    ignore_mismatched_sizes=True  # замена классификационной головы
)

print(f"Модель загружена: {MODEL_NAME}")
print(f"Количество классов: {len(class_names)}")


You passed `num_labels=28` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 200/200 [00:00<00:00, 5367.30it/s]
ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([28])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([28, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Модель загружена: google/vit-base-patch16-224
Количество классов: 28


In [4]:
class SignDataset(Dataset):
    def __init__(self, image_paths, labels, processor):
        self.image_paths = image_paths
        self.labels = labels
        self.processor = processor

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        
        inputs = self.processor(images=image, return_tensors="pt")
        pixel_values = inputs["pixel_values"].squeeze(0)  # [3, 224, 224]

        return {
            "pixel_values": pixel_values,
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [5]:
train_dataset = SignDataset(train_paths, train_labels, processor)
val_dataset   = SignDataset(val_paths,   val_labels,   processor)
test_dataset  = SignDataset(test_paths,  test_labels,  processor)

In [6]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = metric.compute(predictions=predictions, references=labels)
    return accuracy


In [7]:
training_args = TrainingArguments(
    output_dir="./vit-sign-language",
    
    # Эпохи и батчи
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    
    # Оптимизация
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    
    # Логирование и сохранение
    logging_dir="./logs",
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    
    # Прочее
    fp16=torch.cuda.is_available(),  # ускорение на GPU
    dataloader_num_workers=2,
    report_to="none"  # отключить wandb
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [8]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# Обучение
print("Начинаем обучение...")
trainer.train()

# Оценка на тестовой выборке
print("\nОценка на тестовой выборке:")
results = trainer.evaluate(test_dataset)
print(results)


Начинаем обучение...


  2%|▏         | 50/3150 [00:14<12:47,  4.04it/s] 

{'loss': '3.252', 'grad_norm': '6.466', 'learning_rate': '3.048e-05', 'epoch': '0.1587'}


  3%|▎         | 100/3150 [00:27<12:43,  4.00it/s]

{'loss': '1.822', 'grad_norm': '4.404', 'learning_rate': '6.222e-05', 'epoch': '0.3175'}


...

{'train_runtime': '971.2', 'train_samples_per_second': '103.8', 'train_steps_per_second': '3.243', 'train_loss': '0.1241', 'epoch': '10'}


100%|██████████| 3150/3150 [16:13<00:00,  3.24it/s]


Оценка на тестовой выборке:



100%|██████████| 88/88 [00:10<00:00,  8.39it/s]

{'eval_loss': 0.07896158844232559, 'eval_accuracy': 0.9792857142857143, 'eval_runtime': 11.0104, 'eval_samples_per_second': 254.306, 'eval_steps_per_second': 7.992, 'epoch': 10.0}


In [9]:
model.save_pretrained("./vit-sign-final")
processor.save_pretrained("./vit-sign-final")
print("Модель сохранена!")

Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.96s/it]

Модель сохранена!


In [40]:
input_folder = "/home/jupyter/project/letters/"          # папка с исходными изображениями
output_folder = "/home/jupyter/project/test/" # папка для сохранения

os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if filename.endswith((".jpg", ".jpeg", ".png", ".bmp")):
        img_path = os.path.join(input_folder, filename)
        img = Image.open(img_path)
        img_resized = img.resize((224, 224))
        img_resized.save(os.path.join(output_folder, filename))
        print(f"✅ {filename} -> 224x224")

print("Готово!")


✅ A.jpg -> 224x224
✅ B.jpg -> 224x224
✅ C.jpg -> 224x224
✅ D.jpg -> 224x224
✅ E.jpg -> 224x224
✅ F.jpg -> 224x224
✅ G.jpg -> 224x224
✅ H.jpg -> 224x224
✅ I.jpg -> 224x224
✅ J.jpg -> 224x224
✅ K.jpg -> 224x224
✅ L.jpg -> 224x224
✅ M.jpg -> 224x224
✅ N.jpg -> 224x224
✅ O.jpg -> 224x224
✅ P.jpg -> 224x224
✅ Q.jpg -> 224x224
✅ R.jpg -> 224x224
✅ S.jpg -> 224x224
✅ T.jpg -> 224x224
✅ U.jpg -> 224x224
✅ V.jpg -> 224x224
✅ W.jpg -> 224x224
✅ X.jpg -> 224x224
✅ Y.jpg -> 224x224
✅ Z.jpg -> 224x224
✅ space.jpg -> 224x224
Готово!


In [12]:
#The quick brown fox jumps over the lazy dog.

In [41]:
base = "/home/jupyter/project/test"
Lsentence = "The quick brown fox jumps over the lazy dog".upper()
Isentence = []
for i in Lsentence:
    if(i==" "):
        i ="space"
    Isentence.append(base+"/"+i+".jpg")

In [46]:
def predict(image_path, model, processor, class_names):
    model.eval()
    image = Image.open(image_path).convert("RGB")
    
    inputs = processor(images=image, return_tensors="pt")
    
    # Получаем тип модели и устройство
    model_dtype = next(model.parameters()).dtype
    device = next(model.parameters()).device
    
    # Приводим inputs к типу и устройству модели
    inputs = {k: v.to(device=device, dtype=model_dtype) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    probs = torch.softmax(outputs.logits, dim=-1)
    pred_idx = probs.argmax().item()
    confidence = probs[0][pred_idx].item()
    
    return class_names[pred_idx], confidence


prediction = []
for i in Isentence:
    prediction.append(predict(i, model, processor, class_names)[0])

In [51]:
result = ""
for i in prediction:
    if(i =="space"):
        i = " "
    result+=i

In [56]:
def compare_letters(str1, str2):
 # Compare each letter up to the length of the shorter string
    for i in range(min(len(str1), len(str2))):
        if str1[i] < str2[i]:
            print(f"At position {i}: '{str1[i]}' < '{str2[i]}'")
        elif str1[i] > str2[i]:
            print(f"At position {i}: '{str1[i]}' > '{str2[i]}'")
        else:
            print(f"At position {i}: '{str1[i]}' == '{str2[i]}'")
 
     # Handle different lengths
    if len(str1) < len(str2):
        print("str1 is shorter")
    elif len(str1) > len(str2):
        print("str1 is longer")
    else:
        print("Strings are the same length")
compare_letters("result", "Lsentence")

At position 0: 'r' > 'L'
At position 1: 'e' < 's'
At position 2: 's' > 'e'
At position 3: 'u' > 'n'
At position 4: 'l' < 't'
At position 5: 't' > 'e'
str1 is shorter
